# Lembar Kerja Mahasiswa (LKM)
## Praktikum Pertemuan 05 — Regresi Linear

**Cakupan materi:** Linear Regression · Fungsi Basis · Vektorisasi · Error Function · Minimasi Error · Evaluasi Model

---

### Identitas Mahasiswa

*Klik dua kali sel ini untuk mengedit, isi titik-titik di bawah, lalu tekan `Shift + Enter`.*

| | |
|---|---|
| **Nama** | ............................................................ |
| **NIM** | ............................................................ |
| **Kelas / Rombel** | ............................................................ |
| **Nama Dosen** | ............................................................ |
| **Tanggal Praktikum** | ............................................................ |

---

### Capaian yang Diukur

Setelah menyelesaikan lembar kerja ini, Anda diharapkan mampu:

1. Menuliskan model regresi linear dalam notasi vektor/matriks $y = \mathbb{X}\mathbf{w}$.
2. Menerapkan fungsi basis untuk memodelkan hubungan non-linear dengan regresi linear.
3. Membuktikan sendiri manfaat vektorisasi dibandingkan perulangan biasa.
4. Menghitung dan memvisualkan fungsi kesalahan (error function).
5. Menurunkan dan menerapkan persamaan normal untuk meminimalkan kesalahan.
6. Mengevaluasi model dengan MSE dan R² pada data latih maupun data uji, serta mengenali overfitting.


---
## Petunjuk Pengerjaan

**Baca bagian ini sebelum mulai.**

1. Jalankan sel kode **berurutan dari atas ke bawah**. Banyak sel bergantung pada hasil sel sebelumnya.
2. Sel yang berisi tanda `____` atau komentar `# TODO` **harus Anda lengkapi sendiri**. Sel akan error bila dijalankan sebelum dilengkapi — itu normal.
3. Setiap kegiatan memiliki **tabel hasil pengamatan** dan **kotak jawaban** berupa sel markdown. Klik dua kali untuk mengedit, lalu tekan `Shift + Enter`.
4. Angka pada tabel hasil **harus berasal dari eksekusi di komputer Anda sendiri**.
5. Jawaban analisis dinilai dari **penalarannya**, bukan panjangnya.

> **Penting.** Nilai `random_state` diturunkan dari NIM Anda, sehingga hasil setiap mahasiswa akan sedikit berbeda. Jangan menyalin angka orang lain.

### Perkiraan waktu

| Bagian | Perkiraan |
|---|---|
| Persiapan | 5 menit |
| Kegiatan 1–2 (model dasar dan fungsi basis) | 30 menit |
| Kegiatan 3 (vektorisasi) | 15 menit |
| Kegiatan 4–5 (fungsi error dan minimisasi) | 35 menit |
| Kegiatan 6 (evaluasi model) | 25 menit |
| Kesimpulan dan ekspor PDF | 10 menit |

### Cara menyimpan sebagai PDF

Setelah semua sel dijalankan dan seluruh jawaban terisi:

* **JupyterLab:** `File` → `Save and Export Notebook As...` → `HTML`, lalu buka berkas HTML di browser dan cetak (`Ctrl + P`) dengan tujuan **Save as PDF**.
* **Google Colab:** `File` → `Print` → tujuan **Save as PDF**.
* Beri nama berkas: `LKM05_NIM_NamaLengkap.pdf`


---
## Persiapan

Bagian ini **sudah lengkap** — Anda hanya perlu menjalankannya.

> **Catatan.** Seluruh data pada lembar kerja ini dibangkitkan sendiri, sehingga **tidak memerlukan koneksi internet**.

### P.1 Memuat pustaka

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (6, 4.5)
print("Semua pustaka berhasil dimuat.")

### P.2 Mengisi NIM Anda

Ganti angka di bawah dengan **tiga digit terakhir NIM** Anda.

In [ ]:
# TODO (a): ganti dengan tiga digit terakhir NIM Anda, misalnya 137
NIM_3_DIGIT = ____

SEED = int(NIM_3_DIGIT)
print("random_state pribadi Anda (SEED) =", SEED)

### P.3 Membangkitkan data

Kita memakai dua dataset buatan:

| Dataset | Pola sebenarnya | Dipakai pada |
|---|---|---|
| `x_lin, t_lin` | Garis lurus $t = w_0 + w_1 x$ | Kegiatan 1, 4, 5 |
| `x_sin, t_sin` | Gelombang $t = \sin(3\pi x)$ | Kegiatan 2, 3, 6 |


In [ ]:
rng = np.random.default_rng(SEED)

# Dataset 1: hubungan linear murni, w0 = 1 (intersep), w1 = 2 (kemiringan)
N1 = 60
x_lin = np.linspace(0, 5, N1)
W_SEBENARNYA = np.array([1.0, 2.0])       # [w0, w1]
t_lin = W_SEBENARNYA[0] + W_SEBENARNYA[1] * x_lin + rng.normal(0, 1.0, N1)

# Dataset 2: hubungan non-linear (gelombang sinus) berderau
N2 = 70
x_sin = rng.uniform(-1, 1, N2)
x_sin.sort()
t_sin = np.sin(3 * np.pi * x_sin) + rng.normal(0, 0.25, N2)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(x_lin, t_lin, alpha=0.7); ax[0].set_title("Dataset Linear")
ax[0].set_xlabel("x"); ax[0].set_ylabel("t")
ax[1].scatter(x_sin, t_sin, alpha=0.7, color="darkorange"); ax[1].set_title("Dataset Sinus")
ax[1].set_xlabel("x"); ax[1].set_ylabel("t")
plt.tight_layout(); plt.show()

print("Bobot sebenarnya (dataset linear): w0 =", W_SEBENARNYA[0], ", w1 =", W_SEBENARNYA[1])

---
# Kegiatan 1 — Model Regresi Linear dalam Notasi Vektor

**Pertanyaan yang ingin dijawab:** bagaimana model $y = w_0 + w_1 x$ dituliskan sebagai perkalian matriks?

Bentuk vektor/matriksnya:

$$
\mathbf{y} = \mathbb{X}\mathbf{w}, \qquad
\mathbb{X} = \begin{bmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_N \end{bmatrix}, \qquad
\mathbf{w} = \begin{bmatrix} w_0 \\ w_1 \end{bmatrix}
$$

Kolom pertama $\mathbb{X}$ berisi angka 1 semua — inilah yang membuat $w_0$ berperan sebagai intersep.

### 1.1 Menyusun matriks rancangan

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): buat kolom berisi angka 1 sebanyak N1 baris
kolom_bias = np.____((N1, 1))

# TODO (b): gabungkan kolom bias dengan x_lin (ubah dulu menjadi kolom dengan reshape)
X_lin = np.hstack([kolom_bias, x_lin.reshape(-1, ____)])

print("Bentuk X_lin:", X_lin.shape, " (baris = jumlah data, kolom = jumlah parameter)")
print("Lima baris pertama X_lin:\n", X_lin[:5])

### 1.2 Memprediksi dengan perkalian matriks

Alih-alih menulis `w0 + w1*x` untuk tiap titik satu per satu, kita cukup melakukan **satu** perkalian matriks $\mathbb{X}\mathbf{w}$.

In [ ]:
w_contoh = np.array([1.0, 2.0])   # coba dulu dengan bobot sebenarnya

# TODO (c): hitung prediksi dengan perkalian matriks X_lin @ w_contoh
y_contoh = X_lin @ ____

print("5 prediksi pertama:", np.round(y_contoh[:5], 3))
print("5 target asli      :", np.round(t_lin[:5], 3))

plt.scatter(x_lin, t_lin, alpha=0.6, label="Data")
plt.plot(x_lin, y_contoh, color="red", lw=2, label="Prediksi $y=\\mathbb{X}w$")
plt.xlabel("x"); plt.ylabel("t / y"); plt.legend(); plt.title("Model sebagai Perkalian Matriks")
plt.show()

### 1.3 Tabel Hasil Pengamatan

| Aspek | Isian |
|---|---|
| Bentuk (shape) `X_lin` | ...... |
| Isi kolom pertama `X_lin` | ...... |
| Nilai `y_contoh[0]` | ...... |
| Nilai `t_lin[0]` | ...... |

### 1.4 Pertanyaan Analisis

**A1.** Mengapa kolom pertama $\mathbb{X}$ harus berisi angka 1, bukan angka lain? Apa yang terjadi pada $w_0$ bila kolom itu dihilangkan?

> *Jawaban Anda:*
>
> ......

**A2.** Bandingkan `y_contoh` dengan `t_lin`. Keduanya tidak identik padahal `w_contoh` sama persis dengan bobot sebenarnya. Mengapa demikian?

> *Jawaban Anda:*
>
> ......

**A3.** Tuliskan ulang $y = \mathbb{X}\mathbf{w}$ dalam bentuk penjumlahan biasa (tanpa notasi matriks) untuk model dengan **tiga** fitur $x_1, x_2, x_3$.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 2 — Fungsi Basis

**Pertanyaan yang ingin dijawab:** bagaimana regresi *linear* dapat menghasilkan kurva yang melengkung?

**Gagasannya:** kita tidak mengubah modelnya, melainkan **fitur masukannya**. Fungsi basis polinomial mengubah $x$ menjadi $[1, x, x^2, x^3, \ldots]$, lalu regresi linear biasa dijalankan pada fitur yang sudah diperluas itu.

$$
y(x, \mathbf{w}) = w_0 + w_1 x + w_2 x^2 + \dots + w_M x^M
$$

> **Kunci pemahamannya.** Model ini tetap **linear terhadap $\mathbf{w}$**, walaupun kurva $y$ terhadap $x$ melengkung. Itulah sebabnya seluruh teori regresi linear tetap berlaku.

### 2.1 Membandingkan derajat 1 dan derajat tinggi

Lengkapi bagian bertanda `____`.

In [ ]:
def latih_polinomial(x, t, derajat):
    """Melatih regresi linear pada fitur x yang sudah diperluas dengan basis polinomial."""
    # TODO (a): buat objek PolynomialFeatures dengan derajat yang diberikan
    phi = PolynomialFeatures(degree=____, include_bias=True)

    # TODO (b): ubah x (harus berbentuk kolom, gunakan x.reshape(-1, 1)) menjadi fitur polinomial
    X_basis = phi.fit_transform(____.reshape(-1, 1))

    model = LinearRegression(fit_intercept=False)  # intersep sudah ada di X_basis
    model.fit(X_basis, t)
    return model, phi


derajat_uji = [1, 3, 9]
xg = np.linspace(-1, 1, 300)

plt.scatter(x_sin, t_sin, alpha=0.5, color="gray", label="Data")
for d in derajat_uji:
    model_d, phi_d = latih_polinomial(x_sin, t_sin, d)
    yg = model_d.predict(phi_d.transform(xg.reshape(-1, 1)))
    plt.plot(xg, yg, lw=2, label=f"Derajat {d}")
plt.ylim(-2, 2); plt.legend(); plt.title("Fungsi Basis Polinomial pada Berbagai Derajat")
plt.xlabel("x"); plt.ylabel("t / y"); plt.show()

### 2.2 Tabel Hasil Pengamatan

| Derajat | Bentuk kurva (lurus / melengkung wajar / meliuk berlebihan) |
|---|---|
| 1 | ...... |
| 3 | ...... |
| 9 | ...... |

### 2.3 Pertanyaan Analisis

**B1.** Mengapa derajat 1 tidak mampu mengikuti pola data, padahal modelnya tetap disebut "regresi linear"?

> *Jawaban Anda:*
>
> ......

**B2.** Bandingkan derajat 3 dan derajat 9. Derajat mana yang menurut Anda paling *masuk akal* untuk data ini? Kaitkan dengan konsep overfitting dari Pertemuan 03.

> *Jawaban Anda:*
>
> ......

**B3.** Berapa banyak kolom yang dihasilkan `PolynomialFeatures(degree=3)` dari satu fitur $x$? Sebutkan nama tiap kolomnya.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 3 — Vektorisasi

**Pertanyaan yang ingin dijawab:** seberapa besar keuntungan memakai operasi matriks NumPy dibandingkan perulangan `for` biasa?

Kita akan menghitung prediksi $\mathbb{X}\mathbf{w}$ dan galat kuadratnya dengan **dua cara**: memakai perulangan eksplisit, dan memakai operasi tervektorisasi NumPy.

### 3.1 Menyiapkan data berukuran besar

In [ ]:
phi_besar = PolynomialFeatures(degree=5, include_bias=True)
X_besar = phi_besar.fit_transform(x_sin.reshape(-1, 1))
X_besar = np.repeat(X_besar, 80, axis=0)      # diperbanyak agar perbedaan waktunya terasa
t_besar = np.repeat(t_sin, 80)
w_acak = rng.normal(0, 1, X_besar.shape[1])

print("Ukuran X_besar:", X_besar.shape)

### 3.2 Cara 1: dengan perulangan `for`

In [ ]:
t0 = time.perf_counter()

y_loop = np.zeros(X_besar.shape[0])
for i in range(X_besar.shape[0]):
    total = 0.0
    for j in range(X_besar.shape[1]):
        total += X_besar[i, j] * w_acak[j]
    y_loop[i] = total

galat_loop = np.zeros(X_besar.shape[0])
for i in range(X_besar.shape[0]):
    galat_loop[i] = (t_besar[i] - y_loop[i]) ** 2
mse_loop = sum(galat_loop) / len(galat_loop)

waktu_loop = time.perf_counter() - t0
print(f"Waktu dengan perulangan : {waktu_loop:.4f} detik")
print(f"MSE (versi loop)        : {mse_loop:.5f}")

### 3.3 Cara 2: tervektorisasi dengan NumPy

Lengkapi bagian bertanda `____`.

In [ ]:
t0 = time.perf_counter()

# TODO (a): hitung prediksi dengan SATU perkalian matriks, bukan perulangan
y_vek = X_besar @ ____

# TODO (b): hitung MSE dengan operasi tervektorisasi (tanpa perulangan)
#           Petunjuk: (target - prediksi) kuadratkan, lalu rata-ratakan dengan np.mean
mse_vek = np.mean((____ - y_vek) ** 2)

waktu_vek = time.perf_counter() - t0
print(f"Waktu tervektorisasi : {waktu_vek:.6f} detik")
print(f"MSE (versi vektor)   : {mse_vek:.5f}")

print("\nKedua hasil MSE sama?", np.isclose(mse_loop, mse_vek))
print(f"Percepatan: {waktu_loop / waktu_vek:.1f} kali lebih cepat")

### 3.4 Tabel Hasil Pengamatan

| Aspek | Versi loop | Versi vektor |
|---|---|---|
| Waktu eksekusi (detik) | ...... | ...... |
| Nilai MSE | ...... | ...... |
| Berapa kali lipat percepatan | — | ...... |

### 3.5 Pertanyaan Analisis

**C1.** Apakah kedua cara menghasilkan MSE yang sama? Mengapa hasilnya **harus** sama walaupun caranya berbeda?

> *Jawaban Anda:*
>
> ......

**C2.** Berapa kali lipat percepatan yang Anda peroleh? Menurut Anda, apa yang membuat NumPy jauh lebih cepat daripada perulangan Python murni?

> *Jawaban Anda:*
>
> ......

**C3.** Bayangkan dataset ini diperbesar 100 kali lipat lagi. Menurut Anda, versi mana yang waktunya akan bertambah paling drastis? Mengapa vektorisasi menjadi **wajib**, bukan sekadar pilihan, pada data berukuran besar?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 4 — Fungsi Kesalahan (Error Function)

**Pertanyaan yang ingin dijawab:** bagaimana rupa fungsi yang diminimalkan saat kita "melatih" model?

Kita memakai **rata-rata kuadrat galat** (MSE):

$$
E(\mathbf{w}) = \frac{1}{N}\sum_{n=1}^{N}\left(t_n - y(x_n, \mathbf{w})\right)^2 = \frac{1}{N}\|\mathbf{t} - \mathbb{X}\mathbf{w}\|^2
$$

Untuk model dengan dua parameter $(w_0, w_1)$, $E(\mathbf{w})$ dapat digambar sebagai **permukaan tiga dimensi** di atas bidang $(w_0, w_1)$.

### 4.1 Menghitung fungsi error pada sebuah kisi

Lengkapi bagian bertanda `____`.

In [ ]:
def hitung_mse(w0, w1, x, t):
    """Menghitung MSE untuk satu pasang (w0, w1) pada data (x, t)."""
    # TODO (a): hitung prediksi y = w0 + w1*x
    y = ____ + ____ * x
    # TODO (b): hitung rata-rata kuadrat galatnya
    return np.mean((t - y) ** 2)


w0_range = np.linspace(-3, 5, 60)
w1_range = np.linspace(-1, 5, 60)
W0, W1 = np.meshgrid(w0_range, w1_range)

E = np.zeros_like(W0)
for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        E[i, j] = hitung_mse(W0[i, j], W1[i, j], x_lin, t_lin)

idx_min = np.unravel_index(np.argmin(E), E.shape)
w0_terbaik_kisi = W0[idx_min]
w1_terbaik_kisi = W1[idx_min]
print("Bobot terbaik pada kisi : w0 =", round(w0_terbaik_kisi, 3), ", w1 =", round(w1_terbaik_kisi, 3))
print("MSE terkecil pada kisi  :", round(E[idx_min], 4))
print("Bobot sebenarnya        :", W_SEBENARNYA)

### 4.2 Menggambar permukaan error

In [ ]:
fig = plt.figure(figsize=(11, 4.5))

ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(W0, W1, E, cmap="viridis", alpha=0.85)
ax1.set_xlabel("w0"); ax1.set_ylabel("w1"); ax1.set_zlabel("E(w)")
ax1.set_title("Permukaan Error (3D)")

ax2 = fig.add_subplot(1, 2, 2)
kontur = ax2.contour(W0, W1, E, levels=25, cmap="viridis")
ax2.plot(w0_terbaik_kisi, w1_terbaik_kisi, "r*", markersize=18, label="Minimum pada kisi")
ax2.plot(W_SEBENARNYA[0], W_SEBENARNYA[1], "k+", markersize=16, mew=3, label="Bobot sebenarnya")
ax2.set_xlabel("w0"); ax2.set_ylabel("w1"); ax2.set_title("Kontur Error (2D)")
ax2.legend()

plt.tight_layout(); plt.show()

### 4.3 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| $w_0$ terbaik pada kisi | ...... |
| $w_1$ terbaik pada kisi | ...... |
| MSE terkecil pada kisi | ...... |
| $w_0$ sebenarnya | 1,0 |
| $w_1$ sebenarnya | 2,0 |

### 4.4 Pertanyaan Analisis

**D1.** Apakah titik minimum pada kisi ($\star$ merah) berhimpit dengan bobot sebenarnya ($+$ hitam)? Bila tidak persis sama, mengapa itu wajar terjadi?

> *Jawaban Anda:*
>
> ......

**D2.** Perhatikan bentuk kontur pada panel kanan. Apakah bentuknya menyerupai elips, atau ada bentuk lain? Apa yang ditunjukkan oleh kontur yang saling berdekatan dibandingkan yang berjauhan?

> *Jawaban Anda:*
>
> ......

**D3.** Pencarian kisi ini memakai $60 \times 60 = 3.600$ kombinasi untuk **dua** parameter. Bila model memiliki 10 parameter dan tiap parameter dicoba 60 nilai, berapa banyak kombinasi yang diperlukan? Mengapa cara ini tidak praktis untuk model dengan banyak parameter?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 5 — Meminimalkan Fungsi Error (Persamaan Normal)

**Pertanyaan yang ingin dijawab:** adakah cara menemukan titik minimum $E(\mathbf{w})$ tanpa mencoba-coba pada kisi?

Karena $E(\mathbf{w})$ berbentuk kuadratik, titik minimumnya dapat dicari dengan menyamakan **turunannya** dengan nol. Hasilnya dikenal sebagai **persamaan normal**:

$$
\nabla E(\mathbf{w}) = 0 \quad\Longrightarrow\quad \mathbf{w}^\star = (\mathbb{X}^T\mathbb{X})^{-1}\mathbb{X}^T\mathbf{t}
$$

### 5.1 Menerapkan persamaan normal secara manual

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): hitung X^T X
XtX = X_lin.____ @ X_lin

# TODO (b): hitung X^T t
Xtt = X_lin.T @ ____

# TODO (c): selesaikan w = (X^T X)^{-1} X^T t
#           Petunjuk: np.linalg.inv(...) untuk membalik matriks
w_normal = np.linalg.____(XtX) @ Xtt

print("Bobot dari persamaan normal :", np.round(w_normal, 4))
print("Bobot dari pencarian kisi   :", [round(w0_terbaik_kisi, 4), round(w1_terbaik_kisi, 4)])
print("Bobot sebenarnya            :", W_SEBENARNYA)
print("MSE pada w_normal           :", round(hitung_mse(w_normal[0], w_normal[1], x_lin, t_lin), 5))

### 5.2 Membandingkan dengan `np.linalg.lstsq` dan Scikit-Learn

Rumus di atas jarang dipakai langsung dalam praktik karena membalik matriks ($\mathbb{X}^T\mathbb{X}$)$^{-1}$ dapat menjadi tidak stabil secara numerik (ingat kembali ill-conditioning). Alat baku seperti `np.linalg.lstsq` dan `LinearRegression` memakai metode yang lebih stabil, tetapi hasilnya seharusnya sama.

In [ ]:
w_lstsq = np.linalg.lstsq(X_lin, t_lin, rcond=None)[0]

model_sklearn = LinearRegression(fit_intercept=False).fit(X_lin, t_lin)
w_sklearn = model_sklearn.coef_

tabel_bandingkan = pd.DataFrame({
    "Metode": ["Persamaan normal (manual)", "np.linalg.lstsq", "Scikit-Learn"],
    "w0": [w_normal[0], w_lstsq[0], w_sklearn[0]],
    "w1": [w_normal[1], w_lstsq[1], w_sklearn[1]],
}).round(4)
display(tabel_bandingkan)

### 5.3 Tabel Hasil Pengamatan

| Metode | $w_0$ | $w_1$ |
|---|---|---|
| Persamaan normal (manual) | ...... | ...... |
| `np.linalg.lstsq` | ...... | ...... |
| Scikit-Learn | ...... | ...... |
| Pencarian kisi (Kegiatan 4) | ...... | ...... |

### 5.4 Pertanyaan Analisis

**E1.** Apakah keempat cara menghasilkan bobot yang (hampir) sama? Cara mana yang paling **dekat** dengan bobot sebenarnya $[1{,}0,\ 2{,}0]$, dan mengapa tidak ada satu pun yang persis sama?

> *Jawaban Anda:*
>
> ......

**E2.** Bandingkan ketepatan hasil persamaan normal dengan hasil pencarian kisi pada Kegiatan 4. Mengapa persamaan normal bisa lebih tepat padahal komputasinya jauh lebih sedikit?

> *Jawaban Anda:*
>
> ......

**E3.** Pada Pertemuan 07/08 dibahas bahwa membalik $\mathbb{X}^T\mathbb{X}$ dapat bermasalah bila fitur-fiturnya hampir berkolinear. Mengapa masalah itu **tidak muncul** pada data Anda di sini?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 6 — Evaluasi Model

**Pertanyaan yang ingin dijawab:** bagaimana menilai secara jujur apakah sebuah model regresi sudah baik?

**Dua metrik yang dipakai:**

| Metrik | Rumus | Arti | Nilai baik |
|---|---|---|---|
| **MSE** | $\frac{1}{N}\sum(t_n - y_n)^2$ | Rata-rata kuadrat galat | Kecil |
| **R²** | $1 - \frac{\sum(t_n-y_n)^2}{\sum(t_n-\bar{t})^2}$ | Proporsi ragam target yang berhasil dijelaskan | Mendekati 1 |

> **Aturan yang wajib dipegang.** Kedua metrik ini harus dihitung pada **data uji**, bukan data latih — sesuai kaidah dari Pertemuan 03.

### 6.1 Membagi data dan melatih beberapa derajat polinomial

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): bagi data sinus menjadi 70% latih dan 30% uji
x_tr, x_te, t_tr, t_te = train_test_split(x_sin, t_sin, test_size=____, random_state=SEED)

print("Jumlah data latih:", len(x_tr))
print("Jumlah data uji  :", len(x_te))

daftar_derajat = range(1, 15)
hasil_evaluasi = []

for d in daftar_derajat:
    phi = PolynomialFeatures(degree=d, include_bias=True)
    Xtr_d = phi.fit_transform(x_tr.reshape(-1, 1))
    Xte_d = phi.transform(x_te.reshape(-1, 1))

    model = LinearRegression(fit_intercept=False).fit(Xtr_d, t_tr)

    # TODO (b): buat prediksi pada data LATIH dan data UJI
    pred_tr = model.predict(____)
    pred_te = model.predict(____)

    hasil_evaluasi.append({
        "derajat": d,
        "mse_latih": mean_squared_error(t_tr, pred_tr),
        "mse_uji": mean_squared_error(t_te, pred_te),
        "r2_latih": r2_score(t_tr, pred_tr),
        "r2_uji": r2_score(t_te, pred_te),
    })

tabel_evaluasi = pd.DataFrame(hasil_evaluasi).round(4)
display(tabel_evaluasi)

derajat_terbaik = tabel_evaluasi.loc[tabel_evaluasi['mse_uji'].idxmin(), 'derajat']
print("Derajat dengan MSE uji terkecil:", derajat_terbaik)

### 6.2 Menggambar kurva MSE latih vs uji

In [ ]:
plt.plot(tabel_evaluasi["derajat"], tabel_evaluasi["mse_latih"], "o-", label="MSE Latih")
plt.plot(tabel_evaluasi["derajat"], tabel_evaluasi["mse_uji"], "o-", label="MSE Uji")
plt.axvline(derajat_terbaik, ls="--", c="red", alpha=0.6, label=f"Derajat terbaik = {derajat_terbaik}")
plt.yscale("log")
plt.xlabel("Derajat Polinomial"); plt.ylabel("MSE (skala log)")
plt.title("Underfitting vs Overfitting"); plt.legend(); plt.show()

### 6.3 Melihat kurva model terbaik pada data

In [ ]:
phi_terbaik = PolynomialFeatures(degree=int(derajat_terbaik), include_bias=True)
Xtr_terbaik = phi_terbaik.fit_transform(x_tr.reshape(-1, 1))
model_terbaik = LinearRegression(fit_intercept=False).fit(Xtr_terbaik, t_tr)

xg = np.linspace(-1, 1, 300)
yg = model_terbaik.predict(phi_terbaik.transform(xg.reshape(-1, 1)))

plt.scatter(x_tr, t_tr, alpha=0.6, label="Data latih")
plt.scatter(x_te, t_te, alpha=0.6, color="red", marker="s", label="Data uji")
plt.plot(xg, yg, color="black", lw=2, label=f"Model derajat {derajat_terbaik}")
plt.legend(); plt.title("Model Terbaik pada Data Latih dan Uji")
plt.xlabel("x"); plt.ylabel("t"); plt.show()

### 6.4 Tabel Hasil Pengamatan

*Salin dari `tabel_evaluasi` untuk derajat 1, derajat terbaik Anda, dan derajat 12.*

| Derajat | MSE Latih | MSE Uji | R² Latih | R² Uji |
|---|---|---|---|---|
| 1 | ...... | ...... | ...... | ...... |
| ...... (terbaik) | ...... | ...... | ...... | ...... |
| 12 | ...... | ...... | ...... | ...... |

### 6.5 Pertanyaan Analisis

**F1.** Perhatikan kolom MSE Latih dari derajat 1 sampai 12. Apakah nilainya selalu menurun? Mengapa hal itu **tidak boleh** dijadikan dasar memilih derajat terbaik?

> *Jawaban Anda:*
>
> ......

**F2.** Perhatikan kolom MSE Uji. Apakah bentuknya naik-turun-naik (seperti huruf U)? Sebutkan derajat berapa yang menjadi titik terendahnya pada percobaan Anda.

> *Jawaban Anda:*
>
> ......

**F3.** Bandingkan R² Latih dan R² Uji pada derajat 12. Bila R² Latih jauh lebih tinggi daripada R² Uji, apa istilah untuk gejala ini, dan apa penyebabnya?

> *Jawaban Anda:*
>
> ......

**F4.** Berdasarkan seluruh kegiatan pada LKM ini, jelaskan dengan kalimat Anda sendiri **rantai keterkaitan** antara fungsi basis, fungsi error, minimisasi error, dan evaluasi model — dari data mentah sampai model yang siap dipakai.

> *Jawaban Anda:*
>
> ......


---
# Kesimpulan dan Refleksi

### Kesimpulan

*Tuliskan minimal enam poin kesimpulan berdasarkan hasil percobaan Anda sendiri. Sebutkan angka bila relevan.*

1. ......
2. ......
3. ......
4. ......
5. ......
6. ......

### Refleksi

**R1.** Bagian mana dari praktikum ini yang paling sulit Anda pahami? Apa yang akhirnya membuat Anda mengerti, atau apa yang masih mengganjal?

> *Jawaban Anda:*
>
> ......

**R2.** Error apa yang Anda temui saat mengerjakan, dan bagaimana Anda mengatasinya? Sebutkan minimal satu.

> *Jawaban Anda:*
>
> ......

**R3.** Sebelum praktikum ini, mungkin Anda mengira "regresi linear" hanya bisa menghasilkan garis lurus. Bagaimana pemahaman itu berubah setelah mengerjakan Kegiatan 2?

> *Jawaban Anda:*
>
> ......


---
# Rubrik Penilaian

*Bagian ini diisi oleh dosen atau asisten praktikum.*

| No | Aspek yang Dinilai | Bobot | Skor (0–100) | Nilai |
|---|---|---|---|---|
| 1 | Kelengkapan tabel hasil pengamatan (Kegiatan 1–6) | 20% | | |
| 2 | Ketepatan pengisian kode bertanda `TODO` | 25% | | |
| 3 | Kualitas jawaban analisis (A1–F4) | 35% | | |
| 4 | Kesimpulan dan refleksi | 20% | | |
| | **Nilai Akhir** | **100%** | | |

**Catatan dosen:**

> ......

### Pedoman skor jawaban analisis

| Skor | Kriteria |
|---|---|
| 85–100 | Jawaban tepat, didukung angka dari percobaan sendiri, dan menunjukkan penalaran yang jelas |
| 70–84 | Jawaban tepat tetapi kurang didukung data atau penalarannya dangkal |
| 55–69 | Jawaban sebagian benar, atau hanya mengulang teori tanpa mengaitkan hasil percobaan |
| < 55 | Jawaban tidak tepat, kosong, atau merupakan salinan dari mahasiswa lain |


---
# Daftar Periksa Sebelum Mengumpulkan

Centang dengan mengganti `[ ]` menjadi `[x]` (klik dua kali sel ini untuk mengedit).

- [ ] Identitas pada bagian atas sudah diisi lengkap
- [ ] `NIM_3_DIGIT` sudah diganti dengan NIM saya sendiri
- [ ] Semua sel bertanda `TODO` sudah dilengkapi dan berjalan tanpa error
- [ ] Seluruh sel sudah dijalankan berurutan dari atas ke bawah
- [ ] Semua tabel hasil pengamatan sudah diisi angka dari komputer saya
- [ ] Semua pertanyaan analisis A1 sampai F4 sudah dijawab
- [ ] Kesimpulan minimal enam poin dan refleksi R1–R3 sudah ditulis
- [ ] Notebook sudah disimpan (`Ctrl + S`) sebelum diekspor

### Langkah ekspor PDF

1. Simpan notebook: `Ctrl + S`
2. `File` → `Save and Export Notebook As...` → `HTML`
3. Buka berkas HTML di browser, tekan `Ctrl + P`, pilih **Save as PDF**
4. Pada dialog cetak, aktifkan **Background graphics** agar grafik ikut tercetak berwarna
5. Beri nama berkas: `LKM05_NIM_NamaLengkap.pdf`

---

*Lembar Kerja Mahasiswa — Praktikum Pertemuan 05*
*Materi diadaptasi dari kuliah "Linear Regression" oleh Joseph E. Gonzalez dan Narges Norouzi.*
